In [1]:
# input
pdb_cath = "./tmp/pdb-cat.tsv"
pred_cath = "./tmp/single_site_tedId-cath.tsv"
mbp_ids = "./tmp/mbp_files.tsv" # from https://github.com/wangchulab/MetalNet2/blob/main/dataset/collect/cmd.sh, first step
new_mbp_ids = "../../../pdb/collect_mbp/tmp/mbp_files.tsv"
uniprot_to_pdb_ids = "../../collect_annotation/pdb/tmp/entryId-pdbIds.tsv"
uni_site_anno = "../../collect_annotation/uniprot/data/entryId-seqNum-resi-metalResi.tsv"
# output
unknown_cat = "./data/cat-pdbIds-tedRepIds.tsv"

In [2]:
import pandas as pd

df_afdb_anno = pd.read_table(uni_site_anno, header=None, names=["seq_id", "1", "2", "3"], usecols=["seq_id"])
anno_unis = set(df_afdb_anno["seq_id"])

anno_pdb_ids = set()
df_pdb_to_uni = pd.read_table(uniprot_to_pdb_ids, header=None)
chain_to_uni = dict()
for _, row in df_pdb_to_uni.iterrows():
    uni = row[0]
    if uni in anno_unis:
        for i in row[1].split(","):
            anno_pdb_ids.add(str.lower(i[:4]))

mbp_ids = set(pd.read_table(mbp_ids)["pdb"]) | set(pd.read_table(new_mbp_ids)["pdb"]) | anno_pdb_ids

df_pdb = pd.read_table(pdb_cath, header=None, names=["pdb_chain", "cat"])
df_pdb['pdb'] = df_pdb['pdb_chain'].map(lambda x: x[:4])
df_pred = pd.read_table(pred_cath, header=None, names=["ted_id", "cath"])
df_pred['cat'] = df_pred["cath"].map(lambda x: ".".join(x.split(".")[:3]))

df = pd.merge(df_pdb, df_pred, on="cat")
len(set(df['cat']))
len(set(df['cat'].map(lambda x: ".".join(x.split(".")[:2]))))

412

27

In [3]:

mbp_cats = set(df[df['pdb'].map(lambda x: x in mbp_ids)]['cat'])
df = df[df['cat'].map(lambda x: x not in mbp_cats)]
len(mbp_cats)
len(df)
len(set(df['cat']))

264

29936

148

In [4]:
records = []
for (cat, ), df_cat in df.groupby(by=['cat']):
    chains = ",".join(sorted(set(df_cat['pdb_chain'].map(lambda x: f"{x[:4]}_{x[4]}"))))
    teds = ",".join(sorted(set(df_cat['ted_id'])))
    records.append({
        "cat": cat,
        "pdb": chains,
        "ted": teds
    })
pd.DataFrame(records).to_csv(unknown_cat, sep="\t", index=None, header=None)